# Module 4 — Signal Evaluation

This module evaluates the **quality of each characteristic as a
predictive signal** along four dimensions:

- **IC Analysis** — how strongly and consistently does each
   characteristic predict next-month returns?
- **IC Decay** — how quickly does predictive power fade across
   longer horizons?
- **Turnover** — how stable is the signal over time, and what
   does that imply for implementation cost?
- **BH-FDR Correction** — which factors survive rigorous multiple
   testing across all 41 characteristics simultaneously?

**Why M4 complements M2 and M3:**
M2 asked whether a factor is priced on average over 57 years.
M3 asked whether it has independent pricing content.
M4 asks whether the underlying characteristic is a *reliable,
consistent, and implementable* predictor — a factor can pass M2
and M3 but still be unusable if its signal is too noisy or decays
too fast.

**Input:** `outputs/M1/factor_returns_41.parquet` and the raw
CPZ panel (characteristics + returns).

**Outputs** (saved to `outputs/M4/`):
- `ic_results.csv` — Mean IC, ICIR, % positive, BH-FDR results
- `ic_timeseries.parquet` — monthly IC time series for all 41 factors
- `ic_decay.csv` — IC at horizons $h = 1, 2, 3, 6, 12$ months
- `turnover.csv` — monthly rank autocorrelation per factor
- `figures/` — IC bar chart, IC decay curves, ICIR heatmap

In [1]:
# Imports and Configuration

import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import os, warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook"

# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = (r"C:\Users\amosa\Documents\My Graduate School"
                r"\SPRING 2026\Courses\AMS520_Machine Learning in Finance"
                r"\Project\General ML4Trading Resource"
                r"\Personal_Fundamental Factor Investing"
                r"\fundamental-factor-investing")
DATA_DIR = r"C:\Users\amosa\ml4t_data\extended_v2"
M1_OUT   = os.path.join(PROJECT_ROOT, "outputs", "M1")
M2_OUT   = os.path.join(PROJECT_ROOT, "outputs", "M2")
M3_OUT   = os.path.join(PROJECT_ROOT, "outputs", "M3")
OUT_DIR  = os.path.join(PROJECT_ROOT, "outputs", "M4")
FIG_DIR  = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── 41 characteristics by economic family ─────────────────────
CHAR_FAMILIES = {
    "Value":         ["BEME","E2P","CF2P","D2P","S2P","A2ME"],
    "Profitability": ["PROF","ROE","ROA","OP","PM","PCM","RNA"],
    "Investment":    ["Investment","NOA","DPI2A","NI","OA","AC"],
    "Momentum":      ["r12_2","r2_1","r12_7","r36_13",
                      "LT_Rev","SUV","Rel2High"],
    "Risk":          ["Beta","IdioVol","Variance",
                      "Spread","LTurnover","LME"],
    "Other":         ["Q","C","AT","ATO","CTO",
                      "D2A","FC2Y","Lev","SGA2S"],
}
ALL_41 = [c for fam in CHAR_FAMILIES.values() for c in fam]

FAMILY_COLORS = {
    "Value":         "#1f77b4",
    "Profitability": "#2ca02c",
    "Investment":    "#ff7f0e",
    "Momentum":      "#d62728",
    "Risk":          "#9467bd",
    "Other":         "#8c564b",
}

# ── Results from M2 and M3 for cross-referencing ──────────────
M2_PRICED = [
    "BEME","E2P","CF2P","D2P","S2P","A2ME",
    "PROF",
    "Investment","NOA","DPI2A","NI","OA","AC",
    "r2_1","r36_13","LT_Rev","SUV","Rel2High",
    "IdioVol","Variance","Spread","LTurnover","LME",
    "Q","AT","CTO","FC2Y","Lev",
]

M3_NON_SPANNED_PRICED = [
    "BEME","E2P","CF2P","S2P","A2ME",
    "PROF",
    "NOA","NI","OA","AC",
    "r2_1","r36_13","SUV","Rel2High",
    "Variance","Spread","LTurnover","LME",
    "Q","FC2Y",
]

print(f"Output directory: {OUT_DIR}")
print(f"Total factors:    {len(ALL_41)}")
print(f"M2 priced:        {len(M2_PRICED)}/41")
print(f"M3 priced+non-spanned: {len(M3_NON_SPANNED_PRICED)}/41")

Output directory: C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Investing\fundamental-factor-investing\outputs\M4
Total factors:    41
M2 priced:        28/41
M3 priced+non-spanned: 20/41


In [2]:
# Load data

print("Loading data...")

# Factor returns from M1
factor_returns = pd.read_parquet(
    os.path.join(M1_OUT, "factor_returns_41.parquet"))
factor_returns.index = (pd.to_datetime(factor_returns.index)
                        + pd.offsets.MonthEnd(0))
factor_returns = factor_returns[ALL_41].copy()

# Raw panel: characteristics + stock returns
train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
valid = pd.read_parquet(os.path.join(DATA_DIR, "valid.parquet"))
test  = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

panel = pd.concat([train, valid, test], ignore_index=True)
panel['date'] = pd.to_datetime(panel['date']) + pd.offsets.MonthEnd(0)
panel = panel.sort_values(['date','permno']).reset_index(drop=True)

# Load M2 and M3 results for reference
fm_uni   = pd.read_csv(os.path.join(M2_OUT, "lambda_univariate.csv"),
                       index_col=0)
spanning = pd.read_csv(os.path.join(M3_OUT, "spanning_results.csv"),
                       index_col=0)

T = len(factor_returns)
L = int(np.floor(T ** 0.25))

print(f"Factor returns:  {factor_returns.shape}")
print(f"Panel:           {panel.shape[0]:,} stock-month observations")
print(f"T = {T},  Newey-West lag L = {L}")

Loading data...
Factor returns:  (696, 41)
Panel:           1,605,189 stock-month observations
T = 696,  Newey-West lag L = 5


---

## Component 1: Information Coefficient Analysis

The **Information Coefficient (IC)** for factor $j$ in month $t$
is the cross-sectional Spearman rank correlation between the
characteristic values and next-month excess returns:

$$\text{IC}_{j,t} = \text{RankCorr}_i\!\left(
  \tilde{z}_{i,t,j},\; r_{i,t+1}^e\right)$$

We use rank correlation (Spearman) rather than Pearson correlation
because it is robust to outliers in returns - a single stock with
an extreme return in one month will not distort the IC. This is
standard practice in quantitative signal evaluation.

From the $T$ monthly IC values we compute four statistics:

- **Mean IC:** average monthly predictive power. Values above
  $|0.02|$ are considered economically meaningful in the literature.
- **IC volatility ($\sigma_{\text{IC}}$):** standard deviation of
  monthly IC values. High volatility means the signal is
  inconsistent - it works well in some months and poorly in others.
- **ICIR:** $\overline{\text{IC}} / \sigma_{\text{IC}}$, the
  information ratio of the signal. An ICIR above 0.5 (annualized)
  indicates a strong, consistent signal. This is the key metric
  for signal quality.
- **% Positive IC months:** fraction of months where the signal
  pointed in the correct direction. A value well above 50% confirms
  the signal is reliable, not driven by a few extreme months.

Note: the IC computed here uses the **raw characteristic** from
the panel, not the factor return from M1. The sign of the IC
reflects the CPZ coding direction — negative IC for value factors
simply means high $\tilde{z}$ corresponds to growth stocks
(which earn lower returns), consistent with what we found in M2.

In [3]:
# IC Computation

print("Computing Information Coefficients...")
print(f"41 characteristics × {T} months\n")

# Store monthly IC for each factor
ic_records = []
dates = sorted(panel['date'].unique())

for date in dates:
    month_df = panel[panel['date'] == date]

    # Next month's return is already aligned in the panel
    # (ret_excess is the return earned AFTER date t)
    row = {'date': date}
    for char in ALL_41:
        if char not in month_df.columns:
            row[char] = np.nan
            continue
        z   = month_df[char].values
        ret = month_df['ret_excess'].values
        valid = ~np.isnan(z)
        if valid.sum() < 50:
            row[char] = np.nan
            continue
        # Spearman rank correlation
        ic, _ = stats.spearmanr(z[valid], ret[valid])
        row[char] = ic

    ic_records.append(row)

ic_ts = pd.DataFrame(ic_records).set_index('date')
ic_ts.index = pd.to_datetime(ic_ts.index) + pd.offsets.MonthEnd(0)

print(f"IC time series shape: {ic_ts.shape}")

# ── Compute IC summary statistics ─────────────────────────────
ic_results = []
for char in ALL_41:
    s      = ic_ts[char].dropna()
    n      = len(s)
    mean   = s.mean()
    std    = s.std()
    icir   = mean / std * np.sqrt(12) if std > 0 else np.nan
    pct_pos = (s > 0).mean() * 100
    t_ic   = mean / (std / np.sqrt(n)) if std > 0 else np.nan
    # p-value for t-test H0: mean IC = 0
    p_val  = 2 * (1 - stats.t.cdf(abs(t_ic), df=n-1)) if not np.isnan(t_ic) else np.nan

    ic_results.append({
        'Characteristic': char,
        'Family':         next(f for f, cs in CHAR_FAMILIES.items()
                               if char in cs),
        'Mean IC':        round(mean,  5),
        'IC Std':         round(std,   5),
        'ICIR (ann)':     round(icir,  3),
        't (IC)':         round(t_ic,  3) if not np.isnan(t_ic) else np.nan,
        'p-value':        round(p_val, 6) if not np.isnan(p_val) else np.nan,
        '% Pos IC':       round(pct_pos, 1),
        'N months':       n,
    })

ic_df = pd.DataFrame(ic_results).set_index('Characteristic')

print(f"\n{'─'*65}")
print(f"{'Characteristic':<16} {'Family':<15} "
      f"{'Mean IC':>8} {'ICIR':>7} {'%Pos':>6} {'t(IC)':>7}")
print(f"{'─'*65}")
for fam in CHAR_FAMILIES:
    print(f"\n  ── {fam} ──")
    for char in CHAR_FAMILIES[fam]:
        row = ic_df.loc[char]
        sig = ("**" if abs(row['t (IC)']) > 3.00
               else "*" if abs(row['t (IC)']) > 1.96
               else "  ")
        print(f"  {char:<16} {row['Family']:<15} "
              f"{row['Mean IC']:>8.4f} "
              f"{row['ICIR (ann)']:>7.3f} "
              f"{row['% Pos IC']:>5.1f}% "
              f"{row['t (IC)']:>6.3f}{sig}")

n_sig = (ic_df['t (IC)'].abs() > 1.96).sum()
print(f"\n* |t|>1.96   ** |t|>3.00")
print(f"Significant ICs (|t|>1.96): {n_sig}/41")

Computing Information Coefficients...
41 characteristics × 696 months

IC time series shape: (696, 41)

─────────────────────────────────────────────────────────────────
Characteristic   Family           Mean IC    ICIR   %Pos   t(IC)
─────────────────────────────────────────────────────────────────

  ── Value ──
  BEME             Value            -0.1207  -3.677  12.5% -28.000**
  E2P              Value            -0.0203  -0.647  41.7% -4.924**
  CF2P             Value            -0.0419  -1.342  31.3% -10.224**
  D2P              Value             0.0168   0.513  58.1%  3.740**
  S2P              Value            -0.0793  -2.408  21.6% -18.340**
  A2ME             Value            -0.1112  -3.347  13.2% -25.489**

  ── Profitability ──
  PROF             Profitability     0.0117   0.525  56.9%  3.999**
  ROE              Profitability     0.0435   1.503  71.0% 11.445**
  ROA              Profitability     0.0447   1.559  70.4% 11.874**
  OP               Profitability     0.0494  

----

## Component 2: Benjamini-Hochberg FDR Correction

Testing 41 factors simultaneously creates a **multiple testing
problem**. At the 5% significance level we expect
$0.05 \times 41 \approx 2$ false discoveries by chance alone,
even if no factor is truly predictive.

The **Benjamini-Hochberg (1995) False Discovery Rate (BH-FDR)**
procedure controls the expected proportion of false discoveries
among all factors we declare significant. It is less conservative
than Bonferroni (which controls the probability of even one false
discovery) and more appropriate when we expect many true signals.

**The BH procedure:**
1. Sort the 41 $p$-values: $p_{(1)} \leq p_{(2)} \leq \cdots
   \leq p_{(41)}$
2. For FDR threshold $q^* = 0.05$, find the largest $k$ such that:
$$p_{(k)} \leq \frac{k}{41} \times q^*$$
3. Declare all factors with $p$-value $\leq p_{(k)}$ as significant

Factors that survive BH-FDR are significant even after accounting
for the fact that we tested 41 characteristics simultaneously.
This gives us the most defensible set of priced factors for M5.

In [4]:
# BH - FDR

print("=== Benjamini-Hochberg FDR Correction ===\n")
print(f"Testing {len(ALL_41)} factors simultaneously")
print(f"FDR threshold q* = 0.05\n")

# ── BH-FDR procedure ─────────────────────────────────────────
def bh_fdr(p_values, q_star=0.05):
    """
    Benjamini-Hochberg FDR correction.
    Returns boolean array: True = significant after correction.

    Parameters
    ----------
    p_values : array-like of p-values (NaN allowed)
    q_star   : FDR threshold (default 0.05)

    Returns
    -------
    rejected : boolean array, same length as p_values
    threshold: the BH critical p-value
    """
    p   = np.array(p_values, dtype=float)
    n   = len(p)
    # Sort indices by p-value, handling NaN
    nan_mask = np.isnan(p)
    order    = np.argsort(p)         # NaN sorts to end
    p_sorted = p[order]

    # Find largest k where p_(k) <= k/n * q*
    rejected_sorted = np.zeros(n, dtype=bool)
    threshold       = 0.0
    for k in range(n - 1, -1, -1):
        if np.isnan(p_sorted[k]):
            continue
        if p_sorted[k] <= (k + 1) / n * q_star:
            rejected_sorted[:k+1] = True
            threshold = (k + 1) / n * q_star
            break

    # Map back to original order
    rejected = np.zeros(n, dtype=bool)
    rejected[order] = rejected_sorted
    rejected[nan_mask] = False
    return rejected, threshold

p_values  = ic_df['p-value'].values
rejected, bh_threshold = bh_fdr(p_values, q_star=0.05)

ic_df['BH-FDR significant'] = rejected
ic_df['BH threshold']       = bh_threshold

n_bh = rejected.sum()

print(f"BH critical p-value threshold: {bh_threshold:.6f}")
print(f"Factors significant after BH-FDR: {n_bh}/41\n")

print(f"{'─'*70}")
print(f"{'Characteristic':<16} {'p-value':>10} "
      f"{'BH thresh':>10} {'Significant':>12}")
print(f"{'─'*70}")

# Sort by p-value for display
ic_sorted = ic_df.sort_values('p-value')
for char, row in ic_sorted.iterrows():
    sig  = "✓" if row['BH-FDR significant'] else " "
    flag = "**" if row['p-value'] < 0.001 else \
           "*"  if row['p-value'] < 0.05  else "  "
    print(f"  {char:<16} "
          f"{row['p-value']:>10.6f} "
          f"{bh_threshold:>10.6f} "
          f"{sig:>10}  {flag}")

print(f"\n{'─'*70}")
print(f"Factors surviving BH-FDR (q*=0.05): {n_bh}/41")

# Factors that pass naive threshold but fail BH
n_naive = (ic_df['t (IC)'].abs() > 1.96).sum()
n_lost  = n_naive - n_bh
print(f"Factors passing naive |t|>1.96:      {n_naive}/41")
print(f"Lost to multiple testing correction: {n_lost}")

=== Benjamini-Hochberg FDR Correction ===

Testing 41 factors simultaneously
FDR threshold q* = 0.05

BH critical p-value threshold: 0.041463
Factors significant after BH-FDR: 34/41

──────────────────────────────────────────────────────────────────────
Characteristic      p-value  BH thresh  Significant
──────────────────────────────────────────────────────────────────────
  BEME               0.000000   0.041463          ✓  **
  CF2P               0.000000   0.041463          ✓  **
  A2ME               0.000000   0.041463          ✓  **
  S2P                0.000000   0.041463          ✓  **
  ROE                0.000000   0.041463          ✓  **
  PM                 0.000000   0.041463          ✓  **
  OP                 0.000000   0.041463          ✓  **
  ROA                0.000000   0.041463          ✓  **
  RNA                0.000000   0.041463          ✓  **
  Rel2High           0.000000   0.041463          ✓  **
  LT_Rev             0.000000   0.041463          ✓  **
  r36_1

---

## Component 3: IC Decay Analysis

A practically useful signal must maintain predictive power beyond
the immediate next month. We compute the IC at forward horizons
$h = 1, 2, 3, 6, 12$ months:

$$\text{IC}_{j,t}^{(h)} = \text{RankCorr}_i\!\left(
  \tilde{z}_{i,t,j},\; r_{i,t \to t+h}^e\right)$$

where $r_{i,t \to t+h}^e$ is the cumulative excess return from
month $t+1$ to month $t+h$.

**Interpretation:**
- Slow decay (IC remains high at $h=6, 12$): the signal is
  appropriate for lower-turnover strategies. Value factors
  typically show slow decay because book-to-market ratios
  change slowly.
- Fast decay (IC near zero by $h=3$): the signal is only
  exploitable with high-frequency rebalancing. Short-term
  reversal (r2\_1) is an extreme case — by construction its
  signal reverses quickly.
- The **half-life** of a signal is the horizon at which IC
  drops to half its 1-month value. Longer half-life signals
  are cheaper to implement.

In [5]:
# IC decay

print("Computing IC decay across horizons h = 1, 2, 3, 6, 12...")
print("(This computes cumulative forward returns for each horizon)\n")

HORIZONS = [1, 2, 3, 6, 12]

# Build a pivot table: permno × date × ret_excess for fast lookup
# We need cumulative returns h months forward
panel_pivot = panel.pivot(
    index='date', columns='permno', values='ret_excess')
panel_pivot.index = (pd.to_datetime(panel_pivot.index)
                     + pd.offsets.MonthEnd(0))
panel_pivot = panel_pivot.sort_index()

# Cumulative return h months forward:
# cum_ret(t, h) = (1+r_{t+1})(1+r_{t+2})...(1+r_{t+h}) - 1
# We approximate with sum for speed (r << 1)
print("Building forward return matrices...")
fwd_returns = {}
for h in HORIZONS:
    # Sum of next h months of returns (log approximation)
    shifted_sum = pd.DataFrame(0.0,
                               index=panel_pivot.index,
                               columns=panel_pivot.columns)
    for step in range(1, h + 1):
        shifted = panel_pivot.shift(-step)
        shifted_sum = shifted_sum.add(shifted, fill_value=0)
    fwd_returns[h] = shifted_sum
    print(f"  h={h}: done")

# Compute IC at each horizon for each factor
print("\nComputing IC decay...")
decay_records = {char: {} for char in ALL_41}

for h in HORIZONS:
    fwd = fwd_returns[h]
    dates_h = sorted(panel['date'].unique())

    for date in dates_h:
        month_df = panel[panel['date'] == date]
        if date not in fwd.index:
            continue

        fwd_row = fwd.loc[date]

        for char in ALL_41:
            if char not in month_df.columns:
                continue
            z       = month_df[char].values
            permnos = month_df['permno'].values
            fwd_ret = fwd_row.reindex(permnos).values
            valid   = ~(np.isnan(z) | np.isnan(fwd_ret))
            if valid.sum() < 50:
                continue
            ic, _ = stats.spearmanr(z[valid], fwd_ret[valid])
            if h not in decay_records[char]:
                decay_records[char][h] = []
            decay_records[char][h].append(ic)

# Average IC at each horizon
decay_data = []
for char in ALL_41:
    row = {
        'Characteristic': char,
        'Family': next(f for f, cs in CHAR_FAMILIES.items()
                       if char in cs)
    }
    for h in HORIZONS:
        vals = decay_records[char].get(h, [])
        row[f'IC_h{h}'] = round(np.nanmean(vals), 5) if vals else np.nan
    decay_data.append(row)

ic_decay = pd.DataFrame(decay_data).set_index('Characteristic')

print(f"\nIC Decay table (mean IC at each horizon):")
print(f"\n{'Characteristic':<16} {'Family':<15} "
      + "  ".join([f"h={h:>2}" for h in HORIZONS]))
print("─" * 70)
for fam in CHAR_FAMILIES:
    print(f"\n  ── {fam} ──")
    for char in CHAR_FAMILIES[fam]:
        row  = ic_decay.loc[char]
        vals = "  ".join([f"{row[f'IC_h{h}']:>6.3f}"
                          for h in HORIZONS])
        print(f"  {char:<16} {row['Family']:<15} {vals}")

Computing IC decay across horizons h = 1, 2, 3, 6, 12...
(This computes cumulative forward returns for each horizon)

Building forward return matrices...
  h=1: done
  h=2: done
  h=3: done
  h=6: done
  h=12: done

Computing IC decay...

IC Decay table (mean IC at each horizon):

Characteristic   Family          h= 1  h= 2  h= 3  h= 6  h=12
──────────────────────────────────────────────────────────────────────

  ── Value ──
  BEME             Value            0.006   0.014   0.021   0.037   0.061
  E2P              Value            0.049   0.046   0.046   0.046   0.054
  CF2P             Value            0.044   0.045   0.048   0.054   0.071
  D2P              Value            0.042   0.034   0.029   0.021   0.013
  S2P              Value            0.009   0.019   0.026   0.043   0.066
  A2ME             Value            0.002   0.010   0.016   0.031   0.053

  ── Profitability ──
  PROF             Profitability    0.012   0.017   0.020   0.027   0.035
  ROE              Profitabil

In [6]:
# Turnover Analysis

print("Computing signal turnover (rank autocorrelation)...")
print("High rank autocorrelation = low turnover = cheaper to implement\n")

# For each factor and each consecutive month pair,
# compute the rank correlation of characteristic values
# High autocorrelation → signal changes slowly → low turnover

turnover_records = []
dates = sorted(panel['date'].unique())

for char in ALL_41:
    autocorrs = []
    for i in range(1, len(dates)):
        prev_date = dates[i-1]
        curr_date = dates[i]

        prev_df = panel[panel['date'] == prev_date][['permno', char]]
        curr_df = panel[panel['date'] == curr_date][['permno', char]]

        # Merge on common stocks
        merged = prev_df.merge(curr_df, on='permno',
                               suffixes=('_prev','_curr'))
        merged = merged.dropna()
        if len(merged) < 50:
            continue

        ac, _ = stats.spearmanr(
            merged[f'{char}_prev'],
            merged[f'{char}_curr']
        )
        autocorrs.append(ac)

    mean_ac = np.nanmean(autocorrs) if autocorrs else np.nan
    family  = next(f for f, cs in CHAR_FAMILIES.items()
                   if char in cs)

    # Estimate monthly turnover fraction
    # Turnover ≈ fraction of long/short leg that changes each month
    # For rank-normalized chars, turnover ≈ (1 - rank_autocorr) / 2
    turnover_est = (1 - mean_ac) / 2 if not np.isnan(mean_ac) else np.nan

    turnover_records.append({
        'Characteristic':    char,
        'Family':            family,
        'Rank Autocorr':     round(mean_ac,     4),
        'Turnover Est (%)':  round(turnover_est * 100, 1),
    })

turnover_df = pd.DataFrame(turnover_records).set_index('Characteristic')

print(f"{'─'*55}")
print(f"{'Characteristic':<16} {'Family':<15} "
      f"{'Rank AC':>8} {'Turnover%':>10}")
print(f"{'─'*55}")
for fam in CHAR_FAMILIES:
    print(f"\n  ── {fam} ──")
    for char in CHAR_FAMILIES[fam]:
        row = turnover_df.loc[char]
        print(f"  {char:<16} {row['Family']:<15} "
              f"{row['Rank Autocorr']:>8.4f} "
              f"{row['Turnover Est (%)']:>9.1f}%")

# Summary
print(f"\nLowest turnover (most stable signals):")
print(turnover_df.nsmallest(5,'Turnover Est (%)')[
    ['Family','Rank Autocorr','Turnover Est (%)']].to_string())
print(f"\nHighest turnover (fastest-changing signals):")
print(turnover_df.nlargest(5,'Turnover Est (%)')[
    ['Family','Rank Autocorr','Turnover Est (%)']].to_string())

Computing signal turnover (rank autocorrelation)...
High rank autocorrelation = low turnover = cheaper to implement

───────────────────────────────────────────────────────
Characteristic   Family           Rank AC  Turnover%
───────────────────────────────────────────────────────

  ── Value ──
  BEME             Value             0.9805       1.0%
  E2P              Value             0.9563       2.2%
  CF2P             Value             0.9619       1.9%
  D2P              Value             0.9889       0.6%
  S2P              Value             0.9908       0.5%
  A2ME             Value             0.9861       0.7%

  ── Profitability ──
  PROF             Profitability     0.9926       0.4%
  ROE              Profitability     0.9735       1.3%
  ROA              Profitability     0.9769       1.2%
  OP               Profitability     0.9829       0.9%
  PM               Profitability     0.9795       1.0%
  PCM              Profitability     0.9933       0.3%
  RNA              P

---

## Signal Quality Dashboard

We now combine all four M4 components - IC, ICIR, BH-FDR, decay,
and turnover - with the M2 (pricing) and M3 (non-spanning) results
into a single signal quality dashboard.

The evaluation separates two distinct questions:

1. **Should we use this factor?** → Signal score
2. **How costly is it to trade?** → Implementation score

---

### Signal Score

The signal score measures statistical and economic quality.
BH-FDR is a **hard gate** — a factor that fails it cannot be
rated STRONG or MODERATE regardless of its other properties.
For factors that pass BH-FDR, the signal score is a weighted
average of percentile ranks:

$$\text{Signal Score}_j =
  0.50 \cdot \text{rank}(|\text{ICIR}_j|)
+ 0.25 \cdot \mathbf{1}[\text{M2 priced}]
+ 0.25 \cdot \mathbf{1}[\text{M3 non-spanned}]$$

The weights reflect that signal strength (ICIR) matters most,
followed by independent pricing evidence from M2 and M3 equally.
All quantities are converted to percentile ranks (0–100) before
weighting so that the score is on a common scale.

Signal ratings based on the score:

| Rating | Criterion |
|--------|-----------|
| **STRONG** | Passes BH-FDR and signal score $\geq 60$ |
| **MODERATE** | Passes BH-FDR and signal score $30$–$59$ |
| **WEAK** | Fails BH-FDR or signal score $< 30$ |

---

### Implementation Score

The implementation score measures how expensive the factor is
to trade in practice. It is computed independently of the signal
rating and applies to all factors:

$$\text{Impl Score}_j =
  0.60 \cdot \text{rank}(\text{Decay ratio}_{j,h6/h1})
+ 0.40 \cdot \text{rank}(\text{Rank AC}_j)$$

where the decay ratio $= |IC_{h=6}| / |IC_{h=1}|$ measures signal
persistence and rank autocorrelation measures how slowly the
signal changes month to month (a proxy for turnover).

Implementation ratings:

| Rating | Criterion |
|--------|-----------|
| **HIGH** | Implementation score $\geq 60$ — slow-moving, persistent |
| **MEDIUM** | Implementation score $30$–$59$ |
| **LOW** | Implementation score $< 30$ — fast-moving, high turnover |

---

### Interpretation

A factor rated **STRONG/HIGH** is both a reliable predictor and
cheap to trade — the ideal candidate for M5 portfolio construction.

A factor rated **STRONG/LOW** has excellent predictive power but
changes rapidly — it requires high-frequency rebalancing and
incurs significant transaction costs. M5 will apply explicit
turnover constraints to these factors.

A factor rated **MODERATE** or **WEAK** is either statistically
marginal or practically difficult to exploit. These factors are
excluded from the primary M5 portfolio but retained as secondary
candidates for robustness analysis.

In [21]:
# ── Signal quality dashboard ─────────────────
#
# Separates signal quality from implementation quality:
#
# SIGNAL SCORE (determines if the factor is worth using):
#   - BH-FDR:     hard gate (must pass)
#   - |ICIR|:     50% weight
#   - M2 priced:  25% weight  
#   - M3 non-spanned: 25% weight
#
# IMPLEMENTATION SCORE (determines how easy it is to trade):
#   - Decay h6/h1:    60% weight (persistence matters most)
#   - Rank AC:        40% weight (low turnover)
#
# RATING:
#   STRONG:            signal score >= 60
#   MODERATE:          signal score 30-59
#   WEAK:              signal score < 30 or fails BH-FDR
#
# IMPLEMENTATION FLAG (independent of rating):
#   HIGH:   implementation score >= 60
#   MEDIUM: implementation score 30-59
#   LOW:    implementation score < 30

print("=== Signal Quality Dashboard (Revised) ===\n")

W_ICIR_SIG = 0.50
W_M2       = 0.25
W_M3       = 0.25

W_DECAY_IMPL = 0.60
W_AC_IMPL    = 0.40

records = []
for char in ALL_41:
    fam       = next(f for f, cs in CHAR_FAMILIES.items()
                     if char in cs)
    m2_priced = char in M2_PRICED
    m3_ns     = char in M3_NON_SPANNED_PRICED
    m4_bh     = bool(ic_df.loc[char, 'BH-FDR significant'])

    icir      = ic_df.loc[char, 'ICIR (ann)']
    mean_ic   = ic_df.loc[char, 'Mean IC']
    rank_ac   = turnover_df.loc[char, 'Rank Autocorr']
    ic_h1     = ic_decay.loc[char, 'IC_h1']
    ic_h6     = ic_decay.loc[char, 'IC_h6']
    ic_h12    = ic_decay.loc[char, 'IC_h12']

    decay_ratio = np.nan
    if (not np.isnan(ic_h1) and not np.isnan(ic_h6)
            and abs(ic_h1) > 0.001):
        decay_ratio = min(abs(ic_h6) / abs(ic_h1), 1.0)

    records.append({
        'Characteristic': char,
        'Family':         fam,
        'M2 Priced':      m2_priced,
        'M3 Non-spanned': m3_ns,
        'M4 BH-FDR':      m4_bh,
        'ICIR (ann)':     icir,
        'Mean IC':        mean_ic,
        'Decay h6/h1':    decay_ratio,
        'Rank AC':        rank_ac,
        'IC h=1':         ic_h1,
        'IC h=6':         ic_h6,
        'IC h=12':        ic_h12,
    })

dash_df = pd.DataFrame(records).set_index('Characteristic')

# ── Percentile ranks ──────────────────────────────────────────
dash_df['pct_ICIR']  = dash_df['ICIR (ann)'].abs().rank(pct=True) * 100
dash_df['pct_decay'] = dash_df['Decay h6/h1'].rank(
    pct=True, na_option='bottom') * 100
dash_df['pct_ac']    = dash_df['Rank AC'].rank(pct=True) * 100
dash_df['pct_m2']    = dash_df['M2 Priced'].astype(int) * 100
dash_df['pct_m3']    = dash_df['M3 Non-spanned'].astype(int) * 100

# ── Signal score (0-100) ──────────────────────────────────────
# Only computed for BH-FDR survivors — others get 0
dash_df['Signal Score'] = 0.0
bh = dash_df['M4 BH-FDR']
dash_df.loc[bh, 'Signal Score'] = (
    W_ICIR_SIG * dash_df.loc[bh, 'pct_ICIR']
  + W_M2       * dash_df.loc[bh, 'pct_m2']
  + W_M3       * dash_df.loc[bh, 'pct_m3']
)

# ── Implementation score (0-100, all factors) ─────────────────
dash_df['Impl Score'] = (
    W_DECAY_IMPL * dash_df['pct_decay']
  + W_AC_IMPL    * dash_df['pct_ac']
)

# ── Assign signal rating ──────────────────────────────────────
def signal_rating(row):
    if not row['M4 BH-FDR']:
        return 'WEAK'
    if row['Signal Score'] >= 60:
        return 'STRONG'
    if row['Signal Score'] >= 30:
        return 'MODERATE'
    return 'WEAK'

# ── Assign implementation flag ────────────────────────────────
def impl_flag(row):
    if row['Impl Score'] >= 60:
        return 'HIGH'
    if row['Impl Score'] >= 30:
        return 'MEDIUM'
    return 'LOW'

dash_df['Rating']    = dash_df.apply(signal_rating, axis=1)
dash_df['Impl']      = dash_df.apply(impl_flag,     axis=1)

# ── Print dashboard ───────────────────────────────────────────
dash_sorted = dash_df.sort_values(
    ['Rating', 'Signal Score'],
    ascending=[True, False],
    key=lambda x: x.map({'STRONG':0,'MODERATE':1,'WEAK':2})
    if x.name == 'Rating' else x
)

print(f"Signal score weights:  |ICIR|={W_ICIR_SIG:.0%}  "
      f"M2={W_M2:.0%}  M3={W_M3:.0%}")
print(f"Impl score weights:    Decay={W_DECAY_IMPL:.0%}  "
      f"RankAC={W_AC_IMPL:.0%}")
print(f"BH-FDR is a hard gate for STRONG/MODERATE\n")

print(f"{'─'*105}")
print(f"{'Characteristic':<16} {'Fam':<8} "
      f"{'M2':>4} {'M3':>4} {'BH':>4} "
      f"{'ICIR':>7} {'Decay':>6} {'RankAC':>7} "
      f"{'SigScr':>7} {'ImplScr':>8} {'Rating':<10} {'Impl'}")
print(f"{'─'*105}")

for rating_group in ['STRONG', 'MODERATE', 'WEAK']:
    subset = dash_sorted[dash_sorted['Rating'] == rating_group]
    if len(subset) == 0:
        continue
    print(f"\n  ══ {rating_group} ({len(subset)}) ══")
    for char, row in subset.iterrows():
        m2 = '✓' if row['M2 Priced']      else ' '
        m3 = '✓' if row['M3 Non-spanned'] else ' '
        bh = '✓' if row['M4 BH-FDR']      else ' '
        decay_str = (f"{row['Decay h6/h1']:.3f}"
                     if not np.isnan(row['Decay h6/h1'])
                     else "  n/a")
        print(f"  {char:<16} {row['Family'][:7]:<8} "
              f"{m2:>4} {m3:>4} {bh:>4} "
              f"{row['ICIR (ann)']:>7.3f} "
              f"{decay_str:>6} "
              f"{row['Rank AC']:>7.4f} "
              f"{row['Signal Score']:>7.1f} "
              f"{row['Impl Score']:>8.1f} "
              f"{row['Rating']:<10} {row['Impl']}")

n_strong   = (dash_df['Rating'] == 'STRONG').sum()
n_moderate = (dash_df['Rating'] == 'MODERATE').sum()
n_weak     = (dash_df['Rating'] == 'WEAK').sum()

print(f"\n{'─'*105}")
print(f"\nSignal rating summary:")
print(f"  STRONG:   {n_strong}/41")
print(f"  MODERATE: {n_moderate}/41")
print(f"  WEAK:     {n_weak}/41")

print(f"\nImplementation quality among STRONG+MODERATE factors:")
impl_counts = dash_df[dash_df['Rating'].isin(
    ['STRONG','MODERATE'])]['Impl'].value_counts()
for impl, n in impl_counts.items():
    print(f"  {impl}: {n}")

# ── Cross-tabulation: Rating × Implementation ─────────────────
print(f"\n{'─'*55}")
print(f"Rating × Implementation cross-tabulation:")
print(f"{'─'*55}")

for rating_group in ['STRONG', 'MODERATE', 'WEAK']:
    for impl_group in ['HIGH', 'MEDIUM', 'LOW']:
        subset = dash_df[
            (dash_df['Rating'] == rating_group) &
            (dash_df['Impl']   == impl_group)
        ]
        if len(subset) == 0:
            continue
        chars = ', '.join(subset.index.tolist())
        print(f"\n  {rating_group}/{impl_group} ({len(subset)}):")
        print(f"    {chars}")

print(f"\n{'─'*55}")
print(f"Priority for M5:")
print(f"  Tier 1 — STRONG/HIGH:   best candidates")
print(f"  Tier 2 — STRONG/MEDIUM: good candidates")
print(f"  Tier 3 — STRONG/LOW:    use with turnover constraints")
print(f"  Tier 4 — MODERATE:      secondary candidates only")
print(f"  Excluded — WEAK:        do not include in M5")

print(f"\nKey insight: Rating = signal quality (should we use it?)")
print(f"             Impl   = implementation quality (how costly?)")
print(f"             A STRONG/LOW factor is excellent but expensive.")
print(f"             M5 will apply turnover constraints accordingly.")

=== Signal Quality Dashboard (Revised) ===

Signal score weights:  |ICIR|=50%  M2=25%  M3=25%
Impl score weights:    Decay=60%  RankAC=40%
BH-FDR is a hard gate for STRONG/MODERATE

─────────────────────────────────────────────────────────────────────────────────────────────────────────
Characteristic   Fam        M2   M3   BH    ICIR  Decay  RankAC  SigScr  ImplScr Rating     Impl
─────────────────────────────────────────────────────────────────────────────────────────────────────────

  ══ STRONG (18) ══
  Rel2High         Momentu     ✓    ✓    ✓  10.233  1.000  0.8179   100.0     48.0 STRONG     MEDIUM
  SUV              Momentu     ✓    ✓    ✓   4.426  0.504  0.2464    98.8     19.0 STRONG     LOW
  Spread           Risk        ✓    ✓    ✓  -3.881  0.114  0.2451    97.6      3.4 STRONG     LOW
  BEME             Value       ✓    ✓    ✓  -3.677  1.000  0.9805    96.3     64.6 STRONG     HIGH
  Q                Other       ✓    ✓    ✓   3.543  1.000  0.9811    95.1     65.6 STRONG   

In [28]:
# IC Plots

print("Generating IC visualization plots...")

# ── Plot 1: ICIR bar chart ────────────────────────────────────
icir_sorted = ic_df.sort_values('ICIR (ann)', ascending=False)
colors_icir = [FAMILY_COLORS[ic_df.loc[c,'Family']]
               for c in icir_sorted.index]
opacities   = [0.9 if ic_df.loc[c,'BH-FDR significant'] else 0.35
               for c in icir_sorted.index]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x              = icir_sorted.index.tolist(),
    y              = icir_sorted['ICIR (ann)'].values,
    marker_color   = colors_icir,
    marker_opacity = opacities,
    hovertemplate  = "<b>%{x}</b><br>ICIR: %{y:.3f}<extra></extra>"
))
fig1.add_hline(y=0,
               line=dict(color='black', width=1, dash='dash'),
               opacity=0.5)
fig1.add_hline(y=0.5,
               line=dict(color='green', width=1.5, dash='dot'),
               annotation_text="ICIR = 0.5 (strong signal)",
               annotation_position="right")
fig1.add_hline(y=-0.5,
               line=dict(color='green', width=1.5, dash='dot'))
fig1.update_layout(
    title=dict(
        text=("<b>Annualized ICIR by Factor</b><br>"
              "<sup>Sorted by ICIR | Opaque = survives BH-FDR | "
              "Faded = does not survive</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Characteristic',
               tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(title='ICIR (annualized)',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    width=1200, height=480,
    margin=dict(l=60, r=160, t=100, b=130),
)
fig1.show()
fig1.write_image(os.path.join(FIG_DIR, "icir_bar.png"), scale=2)
print("  Saved: icir_bar.png")

# ── Plot 2: IC Decay curves (top 10 by |ICIR|) ───────────────
top10 = ic_df['ICIR (ann)'].abs().nlargest(10).index.tolist()

fig2 = go.Figure()
for char in top10:
    fam   = ic_df.loc[char, 'Family']
    decay_vals = [ic_decay.loc[char, f'IC_h{h}'] for h in HORIZONS]
    fig2.add_trace(go.Scatter(
        x    = HORIZONS,
        y    = decay_vals,
        name = char,
        mode = 'lines+markers',
        line = dict(color=FAMILY_COLORS[fam], width=2),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{char}</b><br>"
            "Horizon: %{x}m<br>"
            "IC: %{y:.4f}<extra></extra>"
        )
    ))
fig2.add_hline(y=0,
               line=dict(color='black', width=1, dash='dash'),
               opacity=0.4)
fig2.update_layout(
    title=dict(
        text=("<b>IC Decay Curves — Top 10 Factors by |ICIR|</b><br>"
              "<sup>Mean cross-sectional rank correlation at each "
              "forward horizon</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Forward Horizon (months)',
               tickvals=HORIZONS),
    yaxis=dict(title='Mean IC',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(orientation='v', yanchor='top', y=1.0,
                xanchor='left', x=1.01, font=dict(size=10)),
    width=900, height=480,
    margin=dict(l=60, r=150, t=80, b=60),
)
fig2.show()
fig2.write_image(os.path.join(FIG_DIR, "ic_decay.png"), scale=2)
print("  Saved: ic_decay.png")

Generating IC visualization plots...


  Saved: icir_bar.png


  Saved: ic_decay.png


In [23]:
print("=== Saving M4 outputs ===\n")

# IC results with BH-FDR
ic_df.to_csv(os.path.join(OUT_DIR, "ic_results.csv"))
print("Saved: ic_results.csv")

# IC time series
ic_ts.to_parquet(os.path.join(OUT_DIR, "ic_timeseries.parquet"))
print("Saved: ic_timeseries.parquet")

# IC decay
ic_decay.to_csv(os.path.join(OUT_DIR, "ic_decay.csv"))
print("Saved: ic_decay.csv")

# Turnover
turnover_df.to_csv(os.path.join(OUT_DIR, "turnover.csv"))
print("Saved: turnover.csv")

# Signal quality dashboard
dash_df.to_csv(os.path.join(OUT_DIR, "signal_dashboard.csv"))
print("Saved: signal_dashboard.csv")

# File inventory
print(f"\nAll M4 output files:")
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:<48} {os.path.getsize(full)/1024:>7.1f} KB")

# Final summary
n_strong   = (dash_df['Rating'] == 'STRONG').sum()
n_moderate = (dash_df['Rating'] == 'MODERATE').sum()
n_weak     = (dash_df['Rating'] == 'WEAK').sum()
n_bh       = ic_df['BH-FDR significant'].sum()

top5_icir  = (ic_df['ICIR (ann)'].abs()
              .nlargest(5)
              .reset_index()
              .to_string(index=False))

# Tier breakdown
def get_tier(rating, impl):
    if rating == 'STRONG'   and impl == 'HIGH':   return 'Tier 1'
    if rating == 'STRONG'   and impl == 'MEDIUM': return 'Tier 2'
    if rating == 'STRONG'   and impl == 'LOW':    return 'Tier 3'
    if rating == 'MODERATE':                       return 'Tier 4'
    return 'Excluded'

dash_df['Tier'] = dash_df.apply(
    lambda r: get_tier(r['Rating'], r['Impl']), axis=1)

tier_summary = ""
for tier, label in [
    ('Tier 1', 'STRONG/HIGH  — best candidates'),
    ('Tier 2', 'STRONG/MEDIUM — good candidates'),
    ('Tier 3', 'STRONG/LOW   — use with turnover constraints'),
    ('Tier 4', 'MODERATE     — secondary candidates only'),
    ('Excluded','WEAK        — excluded from M5'),
]:
    subset = dash_df[dash_df['Tier'] == tier]
    chars  = ', '.join(subset.index.tolist()) if len(subset) else 'none'
    tier_summary += f"  {tier} ({label}): {len(subset)}\n"
    tier_summary += f"    {chars}\n"

print(f"""
{'='*60}
M4 COMPLETE
{'='*60}
Factors evaluated:       41
BH-FDR threshold (q*):  0.05

IC analysis:
  Significant ICs (naive |t|>1.96): {(ic_df['t (IC)'].abs()>1.96).sum()}/41
  Surviving BH-FDR:                 {n_bh}/41

Signal quality ratings:
  STRONG   (BH-FDR + signal score >= 60): {n_strong}/41
  MODERATE (BH-FDR + signal score 30-59): {n_moderate}/41
  WEAK     (fails BH-FDR or score < 30):  {n_weak}/41

Implementation quality (STRONG + MODERATE):
  HIGH:   {((dash_df['Rating'].isin(['STRONG','MODERATE'])) & (dash_df['Impl'] == 'HIGH')).sum()}
  MEDIUM: {((dash_df['Rating'].isin(['STRONG','MODERATE'])) & (dash_df['Impl'] == 'MEDIUM')).sum()}
  LOW:    {((dash_df['Rating'].isin(['STRONG','MODERATE'])) & (dash_df['Impl'] == 'LOW')).sum()}

M5 portfolio tiers:
{tier_summary}
Top 5 factors by |ICIR|:
{top5_icir}

Feeds into:  M5 — Portfolio Construction
{'='*60}
""")

=== Saving M4 outputs ===

Saved: ic_results.csv
Saved: ic_timeseries.parquet
Saved: ic_decay.csv
Saved: turnover.csv
Saved: signal_dashboard.csv

All M4 output files:
  ic_decay.csv                                         2.3 KB
  ic_results.csv                                       3.5 KB
  ic_timeseries.parquet                              286.7 KB
  signal_dashboard.csv                                 8.0 KB
  turnover.csv                                         1.1 KB

M4 COMPLETE
Factors evaluated:       41
BH-FDR threshold (q*):  0.05

IC analysis:
  Significant ICs (naive |t|>1.96): 34/41
  Surviving BH-FDR:                 34/41

Signal quality ratings:
  STRONG   (BH-FDR + signal score >= 60): 18/41
  MODERATE (BH-FDR + signal score 30-59): 10/41
  WEAK     (fails BH-FDR or score < 30):  13/41

Implementation quality (STRONG + MODERATE):
  HIGH:   6
  MEDIUM: 18
  LOW:    4

M5 portfolio tiers:
  Tier 1 (STRONG/HIGH  — best candidates): 6
    BEME, S2P, A2ME, PROF, Q, Lev
  T